# Pipeline Financeiro 2026 — Versão Final Profissional

Notebook único para preparar, validar e exportar o dataset financeiro de 2026.

## Princípios

- dados `raw` nunca são alterados;
- cada regra de negócio é aplicada uma única vez;
- `CODACT 13` é excluído do universo financeiro;
- KILOSPO é calculado integralmente, mas apenas o que associa ao `inform_27` entra no financeiro;
- diferenças Danone são classificadas, não forçadas;
- Setembro permanece no dataset, mas é excluído da reconciliação Danone enquanto estiver incompleto;
- ocupação é calculada ao nível da viagem (`CODEUT`);
- divisões por zero devolvem `NaN`;
- a exportação só ocorre depois das validações finais.

## 00. Configuração

In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform
import duckdb
import numpy as np
import pandas as pd

# ==========================
# 2. Parâmetros
# ==========================
ANO = 2026
MES_VALIDACAO_DANONE_MAX = 9

CUSTO_ESTRUTURA_MENSAL = 103_000.0
EXTRA_DANONE_MENSAL = 20_000.0
DESFASAMENTO_MES_COMBUSTIVEL = -1

CODACT_ESPANHA = [338, 211, 15]
CODACT_EXCLUIR = [13]

TOLERANCIA_FINANCEIRA = 0.01

MAPA_CAPACIDADE = {
    4: 6,
    5: 6,
    6: 6,
    8: 12,
    12: 12,
    14: 14,
    15: 16,
    16: 16,
    18: 18,
    20: 20,
    22: 22,
    24: 24,
    33: 33,
    55: 55,
    66: 66,
}

# ==========================
# 3. Caminhos
# ==========================
def get_paths():
    sistema = platform.system()

    if sistema == "Windows":
        base = Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\01.Financeiro")
        return {
            "db": Path(r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB\2026.duckdb"),
            "excel": base / "Danone_Custos_V3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    if sistema == "Darwin":
        base = Path(
            "/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen"
        )
        return {
            "db": base / "00_DB" / "2026.duckdb",
            "excel": base / "Danone_Custos_v3.xlsx",
            "parquet": base / "inform_27_2026_final.parquet",
            "viagens": base / "viagens_2026.parquet",
            "validacao": base / "validacao_financeira_2026.csv",
            "danone_diag": base / "validacao_danone_2026.csv",
        }

    base = Path.cwd()
    return {
        "db": base / "2026.duckdb",
        "excel": base / "Danone_Custos_v3.xlsx",
        "parquet": base / "inform_27_2026_final.parquet",
        "viagens": base / "viagens_2026.parquet",
        "validacao": base / "validacao_financeira_2026.csv",
        "danone_diag": base / "validacao_danone_2026.csv",
    }

PATHS = get_paths()


## 01. Funções comuns

In [2]:
# ==========================
# 1. Datas
# ==========================
def normalizar_data(serie):
    texto = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )

    data = pd.to_datetime(
        texto,
        format="%Y%m%d",
        errors="coerce",
    )

    mask = data.isna()

    if mask.any():
        data.loc[mask] = pd.to_datetime(
            texto.loc[mask],
            errors="coerce",
        )

    return data

# ==========================
# 2. Referências
# ==========================
def normalizar_referencia(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.extract(r"(\d{10})", expand=False)
        .astype("string")
    )

# ==========================
# 3. Códigos postais
# ==========================
def normalizar_cp(serie):
    cp = (
        serie.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str[:7]
    )

    invalido = (
        cp.str.len().fillna(0).lt(4)
        | cp.eq("0").fillna(False)
        | cp.eq("").fillna(False)
    )

    return cp.mask(invalido, pd.NA)

# ==========================
# 4. Divisão segura
# ==========================
def dividir_seguro(numerador, denominador):
    n = pd.to_numeric(numerador, errors="coerce")
    d = pd.to_numeric(denominador, errors="coerce")
    return n.div(d.where(d.ne(0)))

# ==========================
# 5. Moda segura
# ==========================
def moda_segura(serie):
    serie = serie.dropna()

    if serie.empty:
        return np.nan

    moda = serie.mode()

    if not moda.empty:
        return moda.iloc[0]

    return serie.iloc[0]

## 02. Regras de combustível

In [3]:

# ==========================
# 1. Taxas por CODACT e mês
# ==========================
taxas_ingresso = {
    11: {
        3: 0.0637, 4: 0.1043, 5: 0.0869,
        6: 0.0569, 7: 0.0665, 8: 0.1000,
        9: 0.1229,
    },

    74: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    84: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    118: {
        1: 0.0849, 2: 0.0974, 3: 0.1860,
        4: 0.2434, 5: 0.2188, 6: 0.1763,
        7: 0.1900, 8: 0.2373, 9: 0.2698,
    },

    131: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    262: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    275: {
        3: 0.0669, 4: 0.1077, 5: 0.0902,
        6: 0.0600, 7: 0.0697, 8: 0.1034,
        9: 0.1265,
    },

    280: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    282: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    332: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    391: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    398: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    403: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    404: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    405: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    415: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    418: {
        9: 0.1361,
    },

    419: {
        1: 0.0442, 2: 0.0531, 3: 0.1160,
        4: 0.1568, 5: 0.1393, 6: 0.1091,
        7: 0.1188, 8: 0.1524, 9: 0.1755,
    },

    429: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    588: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },

    660: {
        3: 0.0480, 4: 0.0821, 5: 0.0675,
        6: 0.0422, 7: 0.0504, 8: 0.0785,
        9: 0.0978,
    },

    680: {
        3: 0.0646, 4: 0.1003, 5: 0.0850,
        6: 0.0585, 7: 0.0670, 8: 0.0965,
        9: 0.1167,
    },

    784: {
        3: 0.0753, 4: 0.1170, 5: 0.0991,
        6: 0.0682, 7: 0.0782, 8: 0.1126,
        9: 0.1361,
    },
}


## 03. Importações

In [4]:
# ==========================
# 1. Validar fontes
# ==========================
if not PATHS["db"].exists():
    raise FileNotFoundError(
        f"DuckDB não encontrada: {PATHS['db']}"
    )

if not PATHS["excel"].exists():
    raise FileNotFoundError(
        f"Excel Danone não encontrado: {PATHS['excel']}"
    )

# ==========================
# 2. Ler DuckDB
# ==========================
with duckdb.connect(
    str(PATHS["db"]),
    read_only=True,
) as con:
    df_raw = con.sql("""
        SELECT *
        FROM inform_27_2026
    """).df()

    df_tarifas_raw = con.sql("""
        SELECT
            tipo_local,
            modelo_ingresso,
            tarifa_base,
            data_inicio,
            data_fim
        FROM Danone_Tarifas_2026
    """).df()

    coords_raw = con.sql("""
        SELECT
            cp AS CP,
            point_x AS POINT_X,
            point_y AS POINT_Y
        FROM coordenadas
    """).df()

# ==========================
# 3. Ler KILOSPO
# ==========================
raw_kilospo = pd.read_excel(
    PATHS["excel"],
    sheet_name="KILOSPO",
    header=None,
)

df_kilospo_raw = raw_kilospo.iloc[1:].copy()
df_kilospo_raw.columns = raw_kilospo.iloc[0]

# ==========================
# 4. Validar importações
# ==========================
pd.DataFrame({
    "fonte": [
        "inform_27_2026",
        "KILOSPO",
        "Danone_Tarifas_2026",
        "coordenadas",
    ],
    "linhas": [
        len(df_raw),
        len(df_kilospo_raw),
        len(df_tarifas_raw),
        len(coords_raw),
    ],
})

,fonte,linhas
0,inform_27_2026,817764
1,KILOSPO,44095
2,Danone_Tarifas_2026,37
3,coordenadas,178409


## 04. Universo financeiro — `inform_27_2026`

In [5]:
# ==========================
# 1. Cópia de trabalho
# ==========================
df_base = df_raw.copy()

# ==========================
# 2. Texto
# ==========================
for coluna in df_base.select_dtypes(
    include=["object"]
).columns:
    df_base[coluna] = (
        df_base[coluna]
        .astype("string")
        .str.strip()
    )

# ==========================
# 3. Numéricos
# ==========================
colunas_numericas = [
    "CODACT",
    "INGRESODT",
    "COSTEDT",
    "RENTADT",
    "PALETSDT",
    "PESO_BRUTO",
    "PALETS",
    "KM",
    "KMREALES",
    "CAMION_CAPACIDAD",
]

for coluna in colunas_numericas:
    if coluna in df_base.columns:
        df_base[coluna] = pd.to_numeric(
            df_base[coluna],
            errors="coerce",
        )

# ==========================
# 4. CODEUT
# ==========================
df_base["CODEUT"] = (
    df_base["CODEUT"]
    .astype("string")
    .str.strip()
    .replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA,
    })
)

# ==========================
# 5. Datas
# ==========================
df_base["FCARGA"] = normalizar_data(
    df_base["FCARGA"]
)

df_base["FENTREGA"] = normalizar_data(
    df_base["FENTREGA"]
)

# ==========================
# 6. Chave de referência
# ==========================
df_base["referencia_chave"] = (
    normalizar_referencia(
        df_base["REFERENCIA"]
    )
)

# ==========================
# 7. Guardar universo antes dos filtros
# ==========================
df_raw_check = df_base.copy()

# ==========================
# 8. Ajustes de ingresso
# ==========================
df_base["INGRESODT_ORIGINAL"] = (
    df_base["INGRESODT"]
)

df_base.loc[
    df_base["CODACT"].eq(11),
    "INGRESODT",
] = 0.0

mask_espanha = df_base["CODACT"].isin(
    CODACT_ESPANHA
)

df_base.loc[
    mask_espanha,
    "INGRESODT",
] = (
    df_base.loc[
        mask_espanha,
        "COSTEDT",
    ]
    * 1.05
)

# ==========================
# 9. Excluir CODACT 13
# ==========================
df_base = df_base.loc[
    ~df_base["CODACT"].isin(
        CODACT_EXCLUIR
    )
].copy()

# ==========================
# 10. Filtro operacional
# ==========================
mask_gestao = (
    df_base["GESTION"].eq("LIS")
    | (
        df_base["GESTION"].isna()
        & df_base["PROPIETARIO"].isin(
            ["CEP", "PTG"]
        )
    )
)

df_base = df_base.loc[
    mask_gestao
].copy()

# ==========================
# 11. ID e calendário
# ==========================
df_base["_linha_id"] = np.arange(
    len(df_base),
    dtype="int64",
)

df_base["data"] = df_base["FENTREGA"]
df_base["ano"] = df_base["data"].dt.year
df_base["mes"] = df_base["data"].dt.month
df_base["mes_nome"] = df_base["data"].dt.month_name()
df_base["week_number"] = (
    df_base["data"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)
df_base["week_day"] = (
    df_base["data"]
    .dt.day_name()
)
df_base["mes_estrutura"] = (
    df_base["data"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# ==========================
# 12. Validação
# ==========================
pd.DataFrame({
    "metrica": [
        "Linhas raw",
        "Linhas universo financeiro",
        "CODACT 13 restantes",
        "CODEUT nulos",
        "FENTREGA nula",
    ],
    "valor": [
        len(df_raw),
        len(df_base),
        df_base["CODACT"].eq(13).sum(),
        df_base["CODEUT"].isna().sum(),
        df_base["FENTREGA"].isna().sum(),
    ],
})


,metrica,valor
0,Linhas raw,817764
1,Linhas universo financeiro,126013
2,CODACT 13 restantes,0
3,CODEUT nulos,0
4,FENTREGA nula,0


## 05. Danone — cálculo operacional

A KILOSPO é calculada integralmente.  
O ingresso financeiro só é reconhecido quando existe associação ao universo financeiro.

A diferença entre KILOSPO e `inform_27` é classificada, não forçada.

In [6]:
# ==========================
# 1. Preparar KILOSPO
# ==========================
df_danone = df_kilospo_raw.copy()

df_danone["referencia_chave"] = (
    normalizar_referencia(
        df_danone["PREFPE"]
    )
)

df_danone["PFEENT"] = normalizar_data(
    df_danone["PFEENT"]
)

df_danone["PCODCL"] = (
    df_danone["PCODCL"]
    .astype("string")
    .str.strip()
)

df_danone["tipo_local"] = (
    df_danone["Ruta Tte. Nueva"]
    .astype("string")
    .str.strip()
)

df_danone["Kg Neto Entregado"] = (
    pd.to_numeric(
        df_danone["Kg Neto Entregado"],
        errors="coerce",
    )
)

df_danone = df_danone.loc[
    df_danone["PFEENT"].dt.year.eq(
        ANO
    )
].copy()

# ==========================
# 2. Preparar tarifas
# ==========================
df_tarifas = df_tarifas_raw.copy()

df_tarifas["tipo_local"] = (
    df_tarifas["tipo_local"]
    .astype("string")
    .str.strip()
)

df_tarifas["modelo_ingresso"] = (
    df_tarifas["modelo_ingresso"]
    .astype("string")
    .str.strip()
)

df_tarifas["tarifa_base"] = (
    pd.to_numeric(
        df_tarifas["tarifa_base"],
        errors="coerce",
    )
)

df_tarifas["data_inicio"] = (
    pd.to_datetime(
        df_tarifas["data_inicio"],
        errors="coerce",
    )
)

df_tarifas["data_fim"] = (
    pd.to_datetime(
        df_tarifas["data_fim"],
        errors="coerce",
    )
)

if (
    df_tarifas["tipo_local"].isna().any()
    or df_tarifas["data_inicio"].isna().any()
):
    raise ValueError(
        "Existem tarifas sem tipo_local ou data_inicio."
    )

mask_intervalo_invalido = (
    df_tarifas["data_fim"].notna()
    & df_tarifas["data_fim"].lt(
        df_tarifas["data_inicio"]
    )
)

if mask_intervalo_invalido.any():
    raise ValueError(
        "Existem tarifas com data_fim anterior a data_inicio."
    )

# ==========================
# 3. Validar sobreposição
# ==========================
tarifas_ord = (
    df_tarifas
    .sort_values(
        ["tipo_local", "data_inicio"]
    )
    .copy()
)

tarifas_ord["fim_validacao"] = (
    tarifas_ord["data_fim"]
    .fillna(pd.Timestamp.max.normalize())
)

tarifas_ord["fim_anterior"] = (
    tarifas_ord
    .groupby("tipo_local")[
        "fim_validacao"
    ]
    .shift(1)
)

mask_sobreposicao = (
    tarifas_ord["fim_anterior"].notna()
    & tarifas_ord["data_inicio"].le(
        tarifas_ord["fim_anterior"]
    )
)

if mask_sobreposicao.any():
    tipos = (
        tarifas_ord.loc[
            mask_sobreposicao,
            "tipo_local",
        ]
        .dropna()
        .unique()
        .tolist()
    )
    raise ValueError(
        "Existem períodos tarifários sobrepostos: "
        f"{tipos}"
    )

# ==========================
# 4. Associar tarifa por data
# ==========================
df_danone["modelo_ingresso"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="string",
)

df_danone["tarifa_base"] = pd.Series(
    pd.NA,
    index=df_danone.index,
    dtype="Float64",
)

df_danone["tarifa_data_inicio"] = pd.NaT
df_danone["tarifa_data_fim"] = pd.NaT

tarifa_atribuida = pd.Series(
    False,
    index=df_danone.index,
    dtype="bool",
)

for tarifa in df_tarifas.itertuples(index=False):
    mask_tarifa = (
        df_danone["tipo_local"].eq(
            tarifa.tipo_local
        )
        & df_danone["PFEENT"].ge(
            tarifa.data_inicio
        )
    ).fillna(False)

    if pd.notna(tarifa.data_fim):
        mask_tarifa &= (
            df_danone["PFEENT"].le(
                tarifa.data_fim
            )
            .fillna(False)
        )

    if (tarifa_atribuida & mask_tarifa).any():
        raise ValueError(
            "Mais de uma tarifa aplicável à mesma entrega Danone."
        )

    df_danone.loc[
        mask_tarifa,
        "modelo_ingresso",
    ] = tarifa.modelo_ingresso

    df_danone.loc[
        mask_tarifa,
        "tarifa_base",
    ] = tarifa.tarifa_base

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_inicio",
    ] = tarifa.data_inicio

    df_danone.loc[
        mask_tarifa,
        "tarifa_data_fim",
    ] = tarifa.data_fim

    tarifa_atribuida |= mask_tarifa

# ==========================
# 5. Ingresso base
# ==========================
df_danone["ingresso_danone_base"] = (
    pd.Series(
        pd.NA,
        index=df_danone.index,
        dtype="Float64",
    )
)

mask_ton = (
    df_danone["modelo_ingresso"]
    .eq("POR_TONELADA")
    .fillna(False)
)

mask_entrega = (
    df_danone["modelo_ingresso"]
    .eq("POR_ENTREGA")
    .fillna(False)
)

mask_sem = (
    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
)

df_danone.loc[
    mask_ton,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_ton,
        "Kg Neto Entregado",
    ]
    / 1000
    * df_danone.loc[
        mask_ton,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_entrega,
    "ingresso_danone_base",
] = (
    df_danone.loc[
        mask_entrega,
        "tarifa_base",
    ]
)

df_danone.loc[
    mask_sem,
    "ingresso_danone_base",
] = 0.0

# ==========================
# 6. Extra mensal
# ==========================
df_danone["mes_periodo"] = (
    df_danone["PFEENT"]
    .dt.to_period("M")
)

df_danone["extra_danone"] = 0.0

mask_extra = (
    df_danone["ingresso_danone_base"]
    .notna()
    & df_danone["ingresso_danone_base"]
    .gt(0)
    & df_danone["mes_periodo"]
    .notna()
)

total_base_mes = (
    df_danone["ingresso_danone_base"]
    .where(mask_extra)
    .groupby(
        df_danone["mes_periodo"]
    )
    .transform("sum")
)

if EXTRA_DANONE_MENSAL != 0:
    df_danone.loc[
        mask_extra,
        "extra_danone",
    ] = (
        df_danone.loc[
            mask_extra,
            "ingresso_danone_base",
        ]
        .div(
            total_base_mes.loc[
                mask_extra
            ]
        )
        .mul(EXTRA_DANONE_MENSAL)
    )

df_danone["ingresso_danone"] = (
    df_danone["ingresso_danone_base"]
    + df_danone["extra_danone"]
)

# ==========================
# 7. Estado do cálculo
# ==========================
condicoes = [
    (
        df_danone["modelo_ingresso"]
        .eq("POR_TONELADA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    (
        df_danone["modelo_ingresso"]
        .eq("POR_ENTREGA")
        & df_danone["ingresso_danone"]
        .notna()
    ).fillna(False).to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("SEM_INGRESSO")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("CAPILAR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .eq("POR_DEFINIR")
    .fillna(False)
    .to_numpy(dtype=bool),

    df_danone["modelo_ingresso"]
    .isna()
    .to_numpy(dtype=bool),
]

escolhas = [
    "CALCULADO_POR_TONELADA",
    "CALCULADO_POR_ENTREGA",
    "SEM_INGRESSO",
    "CAPILAR",
    "POR_DEFINIR",
    "SEM_TARIFA",
]

df_danone["estado_calculo"] = (
    np.select(
        condicoes,
        escolhas,
        default="REVER",
    )
)

# ==========================
# 8. Validação mensal
# ==========================
validacao_danone_mes = (
    df_danone
    .groupby(
        "mes_periodo",
        as_index=False,
    )
    .agg(
        linhas=(
            "referencia_chave",
            "size",
        ),
        ingresso_base=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra=(
            "extra_danone",
            "sum",
        ),
        ingresso_final=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
    )
)

validacao_danone_mes


,mes_periodo,linhas,ingresso_base,extra,ingresso_final
0,2026-01,5522,122884.131421,20000.0,142884.131421
1,2026-02,5387,112978.531191,20000.0,132978.531191
2,2026-03,5730,133113.817669,20000.0,153113.817669
3,2026-04,5309,135126.357042,20000.0,155126.357042
4,2026-05,5293,133374.048236,20000.0,153374.048236
5,2026-06,5405,145315.388264,20000.0,165315.388264
6,2026-07,6072,168843.085034,20000.0,188843.085034
7,2026-08,5159,147566.346052,20000.0,167566.346052
8,2026-09,218,3999.145015,20000.0,23999.145015


In [7]:
df_danone

,BLIINF,CODACT,PREFPE,PFEENT,PCODCL,PNOMCL,PRUTA,PCATCL,CODTLI,CODMOP,...,tipo_local,modelo_ingresso,tarifa_base,tarifa_data_inicio,tarifa_data_fim,ingresso_danone_base,mes_periodo,extra_danone,ingresso_danone,estado_calculo
1,FGE50PTG,011,5021758091 413963878,2026-01-02,350150908,PREÇO BAIXO - DOIS AMIGOS,1734,PRE,STD,MAS,...,Porto Prevenda,POR_TONELADA,31.96,2026-01-01,NaT,1.307803,2026-01,0.212851,1.520655,CALCULADO_POR_TONELADA
2,FGE50PTG,011,5021765321 413964675,2026-01-02,350151650,PREÇO BAIXO SUP. AV. FER. AROS,1784,PRE,STD,MAS,...,Porto Prevenda,POR_TONELADA,31.96,2026-01-01,NaT,1.186355,2026-01,0.193085,1.37944,CALCULADO_POR_TONELADA
3,FGE50PTG,011,5021758092 413969049,2026-01-02,350392489,"SANDRA TROVISCO, UNIPESSOAL, L",1734,PRE,STD,MAS,...,Porto Prevenda,POR_TONELADA,31.96,2026-01-01,NaT,0.714626,2026-01,0.116309,0.830934,CALCULADO_POR_TONELADA
4,FGE50PTG,011,5021742353 413966257,2026-01-02,350194778,PREÇO BAIXO - ANGEIRAS,1734,PRE,STD,MAS,...,Porto Prevenda,POR_TONELADA,31.96,2026-01-01,NaT,0.884653,2026-01,0.143982,1.028634,CALCULADO_POR_TONELADA
5,FGE50PTG,011,5021756292 413963116,2026-01-02,350390901,FROIZ - BRAGA II- RETAIL CENTE,1734,PRE,STD,MAS,...,Porto Prevenda,POR_TONELADA,31.96,2026-01-01,NaT,3.466382,2026-01,0.564171,4.030552,CALCULADO_POR_TONELADA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44091,FGE50PTG,011,5023669377 415807061,2026-09-01,350409069,AMOSTRA PROD.DEG.PISO 3,1800,OBS,STD,IMP,...,Ofertas,POR_ENTREGA,28.83029,2026-01-01,NaT,28.83029,2026-09,144.182268,173.012558,CALCULADO_POR_ENTREGA
44092,FGE50PTG,011,5023669958 415811474,2026-09-01,350484542,MIDSID - NOATUM PALMELA,702,DL,STD,444,...,<NA>,<NA>,<NA>,NaT,NaT,<NA>,2026-09,0.000000,<NA>,SEM_TARIFA
44093,FGE50PTG,011,5023692401 415802033,2026-09-01,350476819,AUCHAN - AZAMBUJA,6800,DI2,STD,IT1,...,AUCHAN sin Tte.,SEM_INGRESSO,0.0,2026-01-01,NaT,0.0,2026-09,0.000000,0.0,SEM_INGRESSO
44094,FGE50PTG,011,5023696732 415810653,2026-09-01,350476819,AUCHAN - AZAMBUJA,6800,DI2,STD,IT1,...,AUCHAN sin Tte.,SEM_INGRESSO,0.0,2026-01-01,NaT,0.0,2026-09,0.000000,0.0,SEM_INGRESSO


## 06. Associação Danone e reconciliação

In [8]:
# ==========================
# 1. Agregar por referência/data
# ==========================

danone_chave = (
    df_danone.loc[
        df_danone["referencia_chave"].notna()
        & df_danone["PFEENT"].notna()
    ]
    .groupby(
        [
            "referencia_chave",
            "PFEENT",
        ],
        as_index=False,
    )
    .agg(
        ingresso_danone_total=(
            "ingresso_danone",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        ingresso_danone_base_total=(
            "ingresso_danone_base",
            lambda x: x.sum(
                min_count=1
            ),
        ),
        extra_danone_total=(
            "extra_danone",
            "sum",
        ),
    )
)


# ==========================
# 2. Associar exacto
# ==========================

df = df_base.copy()

df = df.merge(
    danone_chave,
    left_on=[
        "referencia_chave",
        "FENTREGA",
    ],
    right_on=[
        "referencia_chave",
        "PFEENT",
    ],
    how="left",
    validate="many_to_one",
)


# ==========================
# 3. Identificar match
# ==========================

tem_match = (
    df["ingresso_danone_total"]
    .notna()
)


# ==========================
# 4. Identificar destino final
# ==========================

destino_preenchido = (
    df["DESTINO"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

n_destinos = (
    destino_preenchido
    .groupby(
        [
            df["referencia_chave"],
            df["FENTREGA"],
        ],
        dropna=False,
    )
    .transform("sum")
)


# ==========================
# 5. Escolher linha Danone
# ==========================

linha_destino_unico = (
    tem_match
    & destino_preenchido
    & n_destinos.eq(1)
)

linha_fallback = (
    tem_match
    & ~n_destinos.eq(1)
    & ~df.duplicated(
        subset=[
            "referencia_chave",
            "FENTREGA",
        ],
        keep="last",
    )
)

linha_ingresso_danone = (
    linha_destino_unico
    | linha_fallback
)


# ==========================
# 6. Atribuir ingresso Danone
# ==========================

df["ingresso_danone"] = (
    pd.Series(
        pd.NA,
        index=df.index,
        dtype="Float64",
    )
)

df.loc[
    linha_ingresso_danone,
    "ingresso_danone",
] = df.loc[
    linha_ingresso_danone,
    "ingresso_danone_total",
]


# ==========================
# 7. Limpar colunas auxiliares
# ==========================

df = df.drop(
    columns=[
        "PFEENT",
        "ingresso_danone_total",
        "ingresso_danone_base_total",
        "extra_danone_total",
    ],
    errors="ignore",
)


# ==========================
# 8. Ingresso total
# ==========================

df["total_ingresso"] = (
    df["INGRESODT"].fillna(0.0)
    + df["ingresso_danone"].fillna(0.0)
)


In [9]:
# ==========================
# 1. Âmbito de validação
# ==========================
danone_validacao = (
    danone_chave.loc[
        danone_chave["PFEENT"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

base_validacao = (
    df_base.loc[
        df_base["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

raw_validacao = (
    df_raw_check.loc[
        df_raw_check["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
        | df_raw_check["FENTREGA"].isna()
    ]
    .copy()
)

# ==========================
# 2. Chaves disponíveis
# ==========================
chaves_base_exactas = set(
    zip(
        base_validacao["referencia_chave"],
        base_validacao["FENTREGA"],
    )
)

refs_base = set(
    base_validacao[
        "referencia_chave"
    ].dropna()
)

refs_raw = set(
    raw_validacao[
        "referencia_chave"
    ].dropna()
)

refs_codact13 = set(
    raw_validacao.loc[
        raw_validacao["CODACT"].eq(13),
        "referencia_chave",
    ].dropna()
)

# ==========================
# 3. Classificar diferença
# ==========================
def classificar_danone(row):
    chave = (
        row["referencia_chave"],
        row["PFEENT"],
    )

    ref = row["referencia_chave"]

    if chave in chaves_base_exactas:
        return "ASSOCIADO_EXACTO"

    if ref in refs_codact13:
        return "EXCLUIDO_CODACT_13"

    if ref in refs_base:
        return "REFERENCIA_EXISTE_DATA_DIFERENTE"

    if ref in refs_raw:
        return "EXCLUIDO_OUTRO_FILTRO"

    return "FORA_INFORM27"

danone_validacao[
    "estado_reconciliacao"
] = danone_validacao.apply(
    classificar_danone,
    axis=1,
)

# ==========================
# 4. Resumo
# ==========================
validacao_danone = (
    danone_validacao
    .groupby(
        "estado_reconciliacao",
        as_index=False,
    )
    .agg(
        casos=(
            "referencia_chave",
            "size",
        ),
        referencias=(
            "referencia_chave",
            "nunique",
        ),
        ingresso_base=(
            "ingresso_danone_base_total",
            "sum",
        ),
        extra=(
            "extra_danone_total",
            "sum",
        ),
        ingresso=(
            "ingresso_danone_total",
            "sum",
        ),
    )
    .sort_values(
        "ingresso",
        ascending=False,
    )
    .reset_index(drop=True)
)

# ==========================
# 5. Reconciliação financeira real
# ==========================
df_validacao_danone = (
    df.loc[
        df["FENTREGA"]
        .dt.month
        .le(MES_VALIDACAO_DANONE_MAX)
    ]
    .copy()
)

total_danone_df = (
    df_validacao_danone[
        "ingresso_danone"
    ]
    .fillna(0.0)
    .sum()
)

total_danone_associado_exacto = (
    danone_validacao.loc[
        danone_validacao[
            "estado_reconciliacao"
        ].eq("ASSOCIADO_EXACTO"),
        "ingresso_danone_total",
    ]
    .fillna(0.0)
    .sum()
)

erro_reconciliacao_danone = (
    total_danone_df
    - total_danone_associado_exacto
)

# ==========================
# 6. Validar troço final
# ==========================
validacao_troco_danone = (
    df.loc[
        df["referencia_chave"].notna()
        & df["FENTREGA"].notna()
    ]
    .copy()
)

validacao_troco_danone[
    "destino_preenchido"
] = (
    validacao_troco_danone["DESTINO"]
    .astype("string")
    .fillna("")
    .str.strip()
    .ne("")
)

validacao_troco_danone[
    "n_destinos_chave"
] = (
    validacao_troco_danone
    .groupby(
        ["referencia_chave", "FENTREGA"]
    )["destino_preenchido"]
    .transform("sum")
)

casos_troco_danone_errados = int(
    (
        validacao_troco_danone[
            "ingresso_danone"
        ].notna()
        & validacao_troco_danone[
            "n_destinos_chave"
        ].eq(1)
        & ~validacao_troco_danone[
            "destino_preenchido"
        ]
    ).sum()
)

validacao_danone


,estado_reconciliacao,casos,referencias,ingresso_base,extra,ingresso
0,ASSOCIADO_EXACTO,13866,13866,1050288.036634,152965.109168,1203253.145802
1,EXCLUIDO_CODACT_13,24606,24605,41388.505243,6361.791050,47750.296294
2,FORA_INFORM27,2330,2330,9410.699903,20379.869000,29790.568903
3,REFERENCIA_EXISTE_DATA_DIFERENTE,84,84,2112.202415,293.022721,2405.225136


## 07. Combustível

In [10]:
# ==========================
# 1. Mês da taxa
# ==========================
df["mes_taxa_combustivel"] = (
    df["FENTREGA"].dt.month
    + DESFASAMENTO_MES_COMBUSTIVEL
)

# ==========================
# 2. Taxa
# ==========================
def obter_taxa(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return np.nan

    mes = int(mes)

    if mes < 1 or mes > 12:
        return np.nan

    return taxas_ingresso.get(
        int(codact),
        {},
    ).get(
        mes,
        np.nan,
    )


def taxa_configurada_no_periodo(codact, mes):
    if pd.isna(codact) or pd.isna(mes):
        return False

    taxas_codact = taxas_ingresso.get(
        int(codact),
        {},
    )

    if not taxas_codact:
        return False

    mes = int(mes)
    meses = sorted(taxas_codact)

    return (
        min(meses)
        <= mes
        <= max(meses)
    )


df["taxa_combustivel"] = [
    obter_taxa(codact, mes)
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel"] = (
    pd.to_numeric(
        df["taxa_combustivel"],
        errors="coerce",
    )
)

# ==========================
# 3. Base combustível
# ==========================
df["base_combustivel"] = (
    pd.to_numeric(
        df["INGRESODT"],
        errors="coerce",
    )
)

mask_codact11 = (
    df["CODACT"].eq(11)
)

df.loc[
    mask_codact11,
    "base_combustivel",
] = (
    pd.to_numeric(
        df.loc[
            mask_codact11,
            "ingresso_danone",
        ],
        errors="coerce",
    )
)

# ==========================
# 4. Validar taxa em falta
# ==========================
df["taxa_combustivel_esperada"] = [
    taxa_configurada_no_periodo(
        codact,
        mes,
    )
    for codact, mes in zip(
        df["CODACT"],
        df["mes_taxa_combustivel"],
    )
]

df["taxa_combustivel_em_falta"] = (
    df["taxa_combustivel_esperada"]
    & df["taxa_combustivel"].isna()
    & df["base_combustivel"]
    .fillna(0.0)
    .ne(0.0)
)

# ==========================
# 5. Acerto combustível
# ==========================
df["acerto_combustivel"] = 0.0

mask_taxa = (
    df["taxa_combustivel"].notna()
    & df["base_combustivel"].notna()
)

df.loc[
    mask_taxa,
    "acerto_combustivel",
] = (
    df.loc[
        mask_taxa,
        "base_combustivel",
    ]
    * df.loc[
        mask_taxa,
        "taxa_combustivel",
    ]
)

df["total_ingresso_c"] = (
    df["total_ingresso"]
    + df["acerto_combustivel"]
)


## 08. Capacidade e coordenadas

In [11]:
# ==========================
# 1. Capacidade
# ==========================

df["CAMION_CAPACIDAD_NUM"] = (
    pd.to_numeric(
        df["CAMION_CAPACIDAD"],
        errors="coerce",
    )
)

df["capacidade_norm"] = (
    df["CAMION_CAPACIDAD_NUM"]
    .map(MAPA_CAPACIDADE)
)

capacidade_codeut = (
    df.loc[
        df["CODEUT"].notna()
        & df["capacidade_norm"].notna()
    ]
    .groupby("CODEUT")[
        "capacidade_norm"
    ]
    .agg(moda_segura)
)

mask_sem_capacidade = (
    df["CODEUT"].notna()
    & df["capacidade_norm"].isna()
)

df.loc[
    mask_sem_capacidade,
    "capacidade_norm",
] = (
    df.loc[
        mask_sem_capacidade,
        "CODEUT",
    ]
    .map(capacidade_codeut)
)


# ==========================
# 2. Paletes por CODEUT
# ==========================

paletes_codeut = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby("CODEUT")[
        "PALETS"
    ]
    .sum(min_count=1)
)


# ==========================
# 3. Corrigir capacidade 33
# ==========================

df["paletes_codeut"] = (
    df["CODEUT"]
    .map(paletes_codeut)
)

mask_capacidade_33 = (
    df["capacidade_norm"].eq(33)
)

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(40)
    & df["paletes_codeut"].le(55),
    "capacidade_norm",
] = 55

df.loc[
    mask_capacidade_33
    & df["paletes_codeut"].gt(55),
    "capacidade_norm",
] = 66


# ==========================
# 4. Paletes para ocupação
# ==========================

df["paletes_ocupacao_codeut"] = (
    df["paletes_codeut"]
    .clip(upper=66)
)


# ==========================
# 5. Dados válidos
# ==========================

df["dados_validos"] = (
    df["CODEUT"].notna()
    & df["FENTREGA"].notna()
    & df["capacidade_norm"].notna()
    & df["paletes_ocupacao_codeut"].notna()
)


In [12]:
# ==========================
# 1. Preparar coordenadas
# ==========================
coords = coords_raw.copy()

coords["POINT_X"] = pd.to_numeric(
    coords["POINT_X"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["POINT_Y"] = pd.to_numeric(
    coords["POINT_Y"]
    .astype("string")
    .str.strip()
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

coords["CP_chave"] = normalizar_cp(
    coords["CP"]
)

coords["CP_parte"] = (
    coords["CP_chave"]
    .str[:4]
)

df["CPOSTAL_chave"] = normalizar_cp(
    df["CPOSTAL"]
)

df["CPOSTAD_chave"] = normalizar_cp(
    df["CPOSTAD"]
)

df["CPOSTAL_parte"] = (
    df["CPOSTAL_chave"]
    .str[:4]
)

df["CPOSTAD_parte"] = (
    df["CPOSTAD_chave"]
    .str[:4]
)

# ==========================
# 2. CP exacto
# ==========================
coords_cp = (
    coords.dropna(
        subset=[
            "CP_chave",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_chave",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

origem = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAL_chave",
        "POINT_X": "longitude_origem",
        "POINT_Y": "latitude_origem",
    }
)

destino = coords_cp.rename(
    columns={
        "CP_chave": "CPOSTAD_chave",
        "POINT_X": "longitude_destino",
        "POINT_Y": "latitude_destino",
    }
)

df = df.merge(
    origem,
    on="CPOSTAL_chave",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    destino,
    on="CPOSTAD_chave",
    how="left",
    validate="many_to_one",
)

# ==========================
# 3. Fallback 4 dígitos
# ==========================
centroid = (
    coords.dropna(
        subset=[
            "CP_parte",
            "POINT_X",
            "POINT_Y",
        ]
    )
    .groupby(
        "CP_parte",
        as_index=False,
    )[
        ["POINT_X", "POINT_Y"]
    ]
    .mean()
)

centroid_origem = centroid.rename(
    columns={
        "CP_parte": "CPOSTAL_parte",
        "POINT_X": "longitude_origem_fb",
        "POINT_Y": "latitude_origem_fb",
    }
)

centroid_destino = centroid.rename(
    columns={
        "CP_parte": "CPOSTAD_parte",
        "POINT_X": "longitude_destino_fb",
        "POINT_Y": "latitude_destino_fb",
    }
)

df = df.merge(
    centroid_origem,
    on="CPOSTAL_parte",
    how="left",
    validate="many_to_one",
)

df = df.merge(
    centroid_destino,
    on="CPOSTAD_parte",
    how="left",
    validate="many_to_one",
)

df["longitude_origem"] = (
    df["longitude_origem"]
    .fillna(
        df["longitude_origem_fb"]
    )
)

df["latitude_origem"] = (
    df["latitude_origem"]
    .fillna(
        df["latitude_origem_fb"]
    )
)

df["longitude_destino"] = (
    df["longitude_destino"]
    .fillna(
        df["longitude_destino_fb"]
    )
)

df["latitude_destino"] = (
    df["latitude_destino"]
    .fillna(
        df["latitude_destino_fb"]
    )
)

df = df.drop(
    columns=[
        "longitude_origem_fb",
        "latitude_origem_fb",
        "longitude_destino_fb",
        "latitude_destino_fb",
    ],
    errors="ignore",
)

## 09. Custos, viagens e rentabilidade

In [13]:
# ==========================
# 1. Preparar valores
# ==========================
df["PALETS"] = pd.to_numeric(
    df["PALETS"],
    errors="coerce",
).fillna(0.0)

df["COSTEDT"] = pd.to_numeric(
    df["COSTEDT"],
    errors="coerce",
).fillna(0.0)

# ==========================
# 2. Paletes viagem/mês
# ==========================
paletes_codeut_mes = (
    df.groupby(
        [
            "mes_estrutura",
            "CODEUT",
        ],
        dropna=False,
    )["PALETS"]
    .transform("sum")
)

mask_estrutura = (
    df["mes_estrutura"].notna()
    & df["CODEUT"].notna()
    & paletes_codeut_mes.gt(0)
)

# ==========================
# 3. Viagens mensais
# ==========================
n_codeut_mes = (
    df.loc[
        mask_estrutura
    ]
    .groupby(
        "mes_estrutura"
    )["CODEUT"]
    .nunique()
)

df["n_viagens_estrutura_mes"] = (
    df["mes_estrutura"]
    .map(n_codeut_mes)
)

df["custo_estrutura_viagem"] = (
    CUSTO_ESTRUTURA_MENSAL
    / df["n_viagens_estrutura_mes"]
)

# ==========================
# 4. Rateio estrutura
# ==========================
df["custo_estrutura"] = 0.0

df.loc[
    mask_estrutura,
    "custo_estrutura",
] = (
    df.loc[
        mask_estrutura,
        "custo_estrutura_viagem",
    ]
    * df.loc[
        mask_estrutura,
        "PALETS",
    ]
    / paletes_codeut_mes.loc[
        mask_estrutura
    ]
)

# ==========================
# 5. Custo e margem
# ==========================
df["custo_total_c"] = (
    df["COSTEDT"]
    + df["custo_estrutura"]
)

df["margem"] = (
    df["total_ingresso_c"]
    - df["custo_total_c"]
)

df["custo_por_palete"] = (
    dividir_seguro(
        df["custo_total_c"],
        df["PALETS"],
    )
)

df["ingresso_por_palete"] = (
    dividir_seguro(
        df["total_ingresso_c"],
        df["PALETS"],
    )
)

df["margem_por_palete"] = (
    dividir_seguro(
        df["margem"],
        df["PALETS"],
    )
)

df["margem_pct"] = (
    dividir_seguro(
        df["margem"],
        df["total_ingresso_c"],
    )
    * 100
)

In [14]:
# ==========================
# 1. Tabela oficial de viagens
# ==========================
df_viagens = (
    df.loc[
        df["CODEUT"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        data=("FENTREGA", "min"),
        mes_estrutura=(
            "mes_estrutura",
            "min",
        ),
        TRANSPORTISTA=(
            "TRANSPORTISTA",
            "first",
        ),
        LOCORIGEN=(
            "LOCORIGEN",
            "first",
        ),
        LOCDESTINO=(
            "LOCDESTINO",
            "first",
        ),
        paletes=("PALETS", "sum"),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        peso_kg=("PESO_BRUTO", "sum"),
        custo_transporte=(
            "COSTEDT",
            "sum",
        ),
        custo_estrutura=(
            "custo_estrutura",
            "sum",
        ),
        custo_total=(
            "custo_total_c",
            "sum",
        ),
        ingresso_total=(
            "total_ingresso_c",
            "sum",
        ),
        capacidade_original=(
            "CAMION_CAPACIDAD_NUM",
            moda_segura,
        ),
        capacidade_real=(
            "capacidade_norm",
            moda_segura,
        ),
    )
)

# ==========================
# 2. Estado ocupação
# ==========================
condicoes_ocupacao = [
    df_viagens["capacidade_real"]
    .isna()
    .to_numpy(dtype=bool),

    df_viagens["capacidade_real"]
    .le(0)
    .fillna(False)
    .to_numpy(dtype=bool),

    df_viagens["paletes_ocupacao"]
    .isna()
    .to_numpy(dtype=bool),
]

df_viagens["ocupacao_estado"] = (
    np.select(
        condicoes_ocupacao,
        [
            "SEM_CAPACIDADE",
            "CAPACIDADE_INVALIDA",
            "SEM_PALETES",
        ],
        default="OK",
    )
)

# ==========================
# 3. Ocupação
# ==========================
df_viagens["ocupacao_pct"] = (
    dividir_seguro(
        df_viagens["paletes_ocupacao"],
        df_viagens["capacidade_real"],
    )
    * 100
)

df_viagens.loc[
    ~df_viagens[
        "ocupacao_estado"
    ].eq("OK"),
    "ocupacao_pct",
] = np.nan

# ==========================
# 4. KPIs viagem
# ==========================
df_viagens["peso_ton"] = (
    df_viagens["peso_kg"]
    / 1000
)

df_viagens["custo_por_palete"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["paletes"],
    )
)

df_viagens[
    "ingresso_por_palete"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["paletes"],
)

df_viagens["custo_por_ton"] = (
    dividir_seguro(
        df_viagens["custo_total"],
        df_viagens["peso_ton"],
    )
)

df_viagens[
    "ingresso_por_ton"
] = dividir_seguro(
    df_viagens["ingresso_total"],
    df_viagens["peso_ton"],
)

df_viagens["margem"] = (
    df_viagens["ingresso_total"]
    - df_viagens["custo_total"]
)

df_viagens["margem_pct"] = (
    dividir_seguro(
        df_viagens["margem"],
        df_viagens["ingresso_total"],
    )
    * 100
)

# ==========================
# 5. Associar ao detalhe
# ==========================
df = df.merge(
    df_viagens[
        [
            "CODEUT",
            "capacidade_real",
            "ocupacao_pct",
            "ocupacao_estado",
        ]
    ],
    on="CODEUT",
    how="left",
    validate="many_to_one",
)


## 10. Validações finais

In [15]:
# ==========================
# 1. Estrutura mensal
# ==========================
validacao_estrutura = (
    df.loc[
        df["mes_estrutura"].isin(
            n_codeut_mes.index
        )
    ]
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
    )
)

validacao_estrutura["esperado"] = (
    CUSTO_ESTRUTURA_MENSAL
)

validacao_estrutura["erro"] = (
    validacao_estrutura["estrutura"]
    - validacao_estrutura["esperado"]
)

erro_estrutura = (
    validacao_estrutura["erro"]
    .abs()
    .max()
)

if pd.isna(erro_estrutura):
    erro_estrutura = 0.0

# ==========================
# 2. Consistência detalhe/viagens
# ==========================
erro_custo = abs(
    df["custo_total_c"].sum()
    - df_viagens["custo_total"].sum()
)

erro_ingresso = abs(
    df["total_ingresso_c"].sum()
    - df_viagens["ingresso_total"].sum()
)

erro_margem = (
    (
        df["margem"]
        - (
            df["total_ingresso_c"]
            - df["custo_total_c"]
        )
    )
    .abs()
    .max()
)

if pd.isna(erro_margem):
    erro_margem = 0.0

# ==========================
# 3. Validações adicionais
# ==========================
taxas_combustivel_em_falta = int(
    df["taxa_combustivel_em_falta"]
    .sum()
)

linhas_fora_ano = int(
    (
        df["FENTREGA"].notna()
        & ~df["FENTREGA"]
        .dt.year
        .eq(ANO)
    ).sum()
)

# ==========================
# 4. Controlos finais
# ==========================
validacao_financeira = pd.DataFrame({
    "controlo": [
        "Danone df = ASSOCIADO_EXACTO",
        "Danone no troço final",
        "Taxa combustível em falta",
        "FENTREGA fora do ano",
        "Estrutura mensal",
        "Custo detalhe = viagens",
        "Ingresso detalhe = viagens",
        "CODEUT texto nan",
        "CODEUT único em viagens",
        "Ocupação OK > 100%",
        "Margem consistente",
    ],
    "valor": [
        abs(float(erro_reconciliacao_danone)),
        int(casos_troco_danone_errados),
        int(taxas_combustivel_em_falta),
        int(linhas_fora_ano),
        float(erro_estrutura),
        float(erro_custo),
        float(erro_ingresso),
        int(df["CODEUT"].eq("nan").sum()),
        int(
            df_viagens["CODEUT"]
            .duplicated()
            .sum()
        ),
        int(
            df_viagens.loc[
                df_viagens[
                    "ocupacao_estado"
                ].eq("OK"),
                "ocupacao_pct",
            ]
            .gt(100 + 1e-9)
            .sum()
        ),
        float(erro_margem),
    ],
    "tolerancia": [
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        TOLERANCIA_FINANCEIRA,
        0,
        0,
        0,
        TOLERANCIA_FINANCEIRA,
    ],
})

validacao_financeira["estado"] = (
    np.where(
        validacao_financeira["valor"]
        <= validacao_financeira[
            "tolerancia"
        ],
        "OK",
        "ERRO",
    )
)

validacao_financeira


,controlo,valor,tolerancia,estado
0,Danone df = ASSOCIADO_EXACTO,0.000000e+00,0.01,OK
1,Danone no troço final,0.000000e+00,0.00,OK
2,Taxa combustível em falta,0.000000e+00,0.00,OK
3,FENTREGA fora do ano,0.000000e+00,0.00,OK
4,Estrutura mensal,1.455192e-11,0.01,OK
5,Custo detalhe = viagens,9.313226e-10,0.01,OK
6,Ingresso detalhe = viagens,0.000000e+00,0.01,OK
7,CODEUT texto nan,0.000000e+00,0.00,OK
8,CODEUT único em viagens,0.000000e+00,0.00,OK
9,Ocupação OK > 100%,2.318000e+03,0.00,ERRO


## 11. Dataset final e exportação

In [16]:
# ==========================
# 1. Colunas finais
# ==========================
colunas_finais = [
    "REFERENCIA",
    "referencia_chave",
    "FENTREGA",
    "FCARGA",
    "CODEUT",
    "capacidade_norm",
    "paletes_codeut",
    "paletes_ocupacao_codeut",
    "capacidade_real",
    "ocupacao_pct",
    "ocupacao_estado",
    "PALETS",
    "PESO_BRUTO",
    "INGRESODT",
    "ingresso_danone",
    "total_ingresso",
    "taxa_combustivel",
    "taxa_combustivel_esperada",
    "taxa_combustivel_em_falta",
    "base_combustivel",
    "acerto_combustivel",
    "total_ingresso_c",
    "COSTEDT",
    "custo_estrutura",
    "custo_total_c",
    "margem",
    "margem_pct",
    "custo_por_palete",
    "ingresso_por_palete",
    "margem_por_palete",
    "PROV_ORIGEN",
    "LOCORIGEN",
    "PROV_DESTINO",
    "LOCDESTINO",
    "CPOSTAL",
    "CPOSTAD",
    "longitude_origem",
    "latitude_origem",
    "longitude_destino",
    "latitude_destino",
    "TRANSPORTISTA",
    "week_day",
    "week_number",
    "ano",
    "mes",
    "mes_nome",
    "data",
    "mes_estrutura",
    "dados_validos",
    "LOCCAR",
    "LUGARCARGA",
    "LOCDES",
    "LUGARDESCARGA",
    "TRACTORA",
    "CODEDT",
    "CODACT",
    "TIPOCLIENTE",
    "TIPOFLUJO",
    "PROV_ENTREGAR",
    "PAISENTREGAR",
    "ACTIVIDAD",
]

colunas_existentes = [
    coluna
    for coluna in colunas_finais
    if coluna in df.columns
]

df_final = (
    df[colunas_existentes]
    .copy()
)

df_final.head()


,REFERENCIA,referencia_chave,FENTREGA,FCARGA,CODEUT,capacidade_norm,paletes_codeut,paletes_ocupacao_codeut,capacidade_real,ocupacao_pct,...,LOCDES,LUGARDESCARGA,TRACTORA,CODEDT,CODACT,TIPOCLIENTE,TIPOFLUJO,PROV_ENTREGAR,PAISENTREGAR,ACTIVIDAD
0,5021950798 414190635,5021950798,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319558,3. MALAQUIAS - CASH & CARRY O. AZ,93-QI-75,25531441,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
1,2021792046,2021792046,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,298515,DL Transportes Fernando Simões Monteiro Unip. Lda,93-QI-75,25532252,757,<NA>,Directo,Coimbra,PORTUGAL,SUMOLCOMPAL MARKETING
2,5021982094 414203894,5021982094,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319783,"3. MARABUTO-PRODUT.ALIMENTARES,SA",93-QI-75,25532753,11,TLD PORTUGAL,Directo,Aveiro,PORTUGAL,DANONE PORTUGAL
3,5021963392 414182550,5021963392,2026-02-02,2026-02-01,3365589,33.0,25.0,25.0,33.0,75.757576,...,319201,3. COOPERATIVA AGRICOLA DA TOCHA,93-QI-75,25534995,11,TLD PORTUGAL,Directo,Coimbra,PORTUGAL,DANONE PORTUGAL
4,5021973560 414166769,5021973560,2026-02-02,2026-01-30,3365590,33.0,35.0,35.0,33.0,106.060606,...,328547,DL TRANSPARENTODISSEIA - Transportes Unipessoa...,0000XXX,25515326,11,TLD PORTUGAL,Directo,Faro,PORTUGAL,DANONE PORTUGAL


In [17]:

# ==========================
# 1. Validar Danone no output
# ==========================
validacao = df.loc[
    df["mes"].eq(9)
].groupby("ACTIVIDAD", dropna=False).agg(
    ingresso_danone=("ingresso_danone", "sum"),
    ingresso_total=("total_ingresso", "sum"),
).reset_index()

display(
    validacao.loc[
        validacao["ACTIVIDAD"].astype("string").str.contains(
            "DANONE", case=False, na=False
        )
    ]
)


,ACTIVIDAD,ingresso_danone,ingresso_total
11,DANONE PORTUGAL,0.0,0.0
12,DANONE PORTUGAL TLD,0.0,0.0


In [18]:
# ==========================
# 1. Exportar detalhe
# ==========================
df_final.to_parquet(
    PATHS["parquet"],
    index=False,
)

# ==========================
# 2. Exportar viagens
# ==========================
df_viagens.to_parquet(
    PATHS["viagens"],
    index=False,
)

# ==========================
# 3. Exportar validações
# ==========================
validacao_financeira.to_csv(
    PATHS["validacao"],
    index=False,
    sep=";",
    decimal=",",
)

validacao_danone.to_csv(
    PATHS["danone_diag"],
    index=False,
    sep=";",
    decimal=",",
)

# ==========================
# 4. Resultado
# ==========================
pd.DataFrame({
    "output": [
        "Detalhe Power BI",
        "Viagens",
    ],
    "ficheiro": [
        str(PATHS["parquet"]),
        str(PATHS["viagens"]),
    ],
    "linhas": [
        len(df_final),
        len(df_viagens),
    ],
})

,output,ficheiro,linhas
0,Detalhe Power BI,/Users/rr/Library/Mobile Documents/com~apple~C...,126013
1,Viagens,/Users/rr/Library/Mobile Documents/com~apple~C...,19754


# 12. Análises

A partir deste ponto não são alteradas regras de negócio nem valores do pipeline.  
As análises usam apenas `df_final`, `df_viagens` e as tabelas de validação já produzidas.

In [19]:
# ==========================
# 1. Resumo financeiro mensal
# ==========================
resumo_mensal = (
    df_final
    .groupby(
        "mes_estrutura",
        as_index=False,
    )
    .agg(
        linhas=("CODEUT", "size"),
        viagens=("CODEUT", "nunique"),
        paletes=("PALETS", "sum"),
        peso_kg=("PESO_BRUTO", "sum"),
        ingresso=(
            "total_ingresso_c",
            "sum",
        ),
        custo=(
            "custo_total_c",
            "sum",
        ),
        margem=("margem", "sum"),
        estrutura=(
            "custo_estrutura",
            "sum",
        ),
        combustivel=(
            "acerto_combustivel",
            "sum",
        ),
    )
)

resumo_mensal["peso_ton"] = (
    resumo_mensal["peso_kg"]
    / 1000
)

resumo_mensal["margem_pct"] = (
    dividir_seguro(
        resumo_mensal["margem"],
        resumo_mensal["ingresso"],
    )
    * 100
)

resumo_mensal

,mes_estrutura,linhas,viagens,paletes,peso_kg,ingresso,custo,margem,estrutura,combustivel,peso_ton,margem_pct
0,2026-01-01,11327,1923,46298.001,10473868.763,464356.71835,513737.91,-49381.19165,103000.0,0.000000,10473.868763,-10.634323
1,2026-02-01,10645,1718,38238.75,8952691.761,411041.180994,458867.4,-47826.219006,103000.0,2058.953390,8952.691761,-11.635384
2,2026-03-01,12281,1966,46487.563,12553858.055,508804.760301,540931.2,-32126.439699,103000.0,2926.743206,12553.858055,-6.3141
3,2026-04-01,12577,1972,48077.75,11656719.178,558217.411672,572365.89,-14148.478328,103000.0,30094.108710,11656.719178,-2.534582
4,2026-05-01,12581,1953,47682.05,11801330.019,564403.313304,573870.16,-9466.846696,103000.0,45637.587756,11801.330019,-1.677319
5,2026-06-01,12460,1966,45447.16,11952113.721,551492.711347,580716.09,-29223.378653,103000.0,40162.106093,11952.113721,-5.29896
6,2026-07-01,14179,2167,49278.5,13429174.515,612728.697998,643897.92,-31169.222002,103000.0,31608.765802,13429.174515,-5.086953
7,2026-08-01,13273,2019,49250.0,12551971.754,594792.832797,607085.81,-12292.977203,103000.0,34858.547003,12551.971754,-2.066766
8,2026-09-01,12754,1979,46901.0,12164160.39,0.0,103000.0,-103000.0,103000.0,0.000000,12164.16039,<NA>
9,2026-10-01,13518,1990,46723.18,12286748.087,0.0,103000.0,-103000.0,103000.0,0.000000,12286.748087,<NA>


In [20]:
# ==========================
# 1. KPIs de viagens
# ==========================
viagens_validas = (
    df_viagens.loc[
        df_viagens[
            "ocupacao_estado"
        ].eq("OK")
    ]
    .copy()
)

kpi_viagens = pd.DataFrame({
    "metrica": [
        "Viagens",
        "Paletes",
        "Peso ton",
        "Ocupação média %",
        "Custo por viagem",
        "Custo por palete",
        "Custo por ton",
        "Margem",
    ],
    "valor": [
        df_viagens["CODEUT"].nunique(),
        df_viagens["paletes"].sum(),
        df_viagens["peso_ton"].sum(),
        viagens_validas[
            "ocupacao_pct"
        ].mean(),
        df_viagens[
            "custo_total"
        ].mean(),
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "paletes"
                ].sum()
            ]),
        ).iloc[0],
        dividir_seguro(
            pd.Series([
                df_viagens[
                    "custo_total"
                ].sum()
            ]),
            pd.Series([
                df_viagens[
                    "peso_ton"
                ].sum()
            ]),
        ).iloc[0],
        df_viagens["margem"].sum(),
    ],
})

kpi_viagens

,metrica,valor
0,Viagens,19754.000000
1,Paletes,465201.954000
2,Peso ton,118032.819864
3,Ocupação média %,76.497010
4,Custo por viagem,243.012675
5,Custo por palete,10.319115
6,Custo por ton,40.670657
7,Margem,-534634.753239


In [21]:

# ==========================
# 1. Viagens únicas
# ==========================

validacao_enchimento = (
    df.loc[
        df["dados_validos"]
        & df["TRANSPORTISTA"].notna()
    ]
    .groupby(
        "CODEUT",
        as_index=False,
    )
    .agg(
        transportador=(
            "TRANSPORTISTA",
            "first",
        ),
        paletes=(
            "paletes_codeut",
            "first",
        ),
        paletes_ocupacao=(
            "paletes_ocupacao_codeut",
            "first",
        ),
        capacidade=(
            "capacidade_norm",
            "first",
        ),
    )
)


# ==========================
# 2. Taxa por viagem
# ==========================

validacao_enchimento["taxa_enchimento"] = (
    validacao_enchimento["paletes_ocupacao"]
    / validacao_enchimento["capacidade"]
)

validacao_enchimento["taxa_enchimento_pct"] = (
    validacao_enchimento["taxa_enchimento"]
    * 100
)


# ==========================
# 3. Resumo por transportador
# ==========================

enchimento_transportador = (
    validacao_enchimento
    .groupby(
        "transportador",
        as_index=False,
    )
    .agg(
        viagens=(
            "CODEUT",
            "nunique",
        ),
        paletes=(
            "paletes_ocupacao",
            "sum",
        ),
        capacidade_total=(
            "capacidade",
            "sum",
        ),
        taxa_media_viagem_pct=(
            "taxa_enchimento_pct",
            "mean",
        ),
    )
)


# ==========================
# 4. Taxa ponderada
# ==========================

enchimento_transportador[
    "taxa_enchimento_pct"
] = (
    enchimento_transportador["paletes"]
    / enchimento_transportador["capacidade_total"]
    * 100
)


# ==========================
# 5. Formatar
# ==========================

enchimento_transportador = (
    enchimento_transportador
    .sort_values(
        "taxa_enchimento_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

enchimento_transportador[
    [
        "taxa_media_viagem_pct",
        "taxa_enchimento_pct",
    ]
] = (
    enchimento_transportador[
        [
            "taxa_media_viagem_pct",
            "taxa_enchimento_pct",
        ]
    ]
    .round(2)
)


# ==========================
# 6. Resultado
# ==========================

enchimento_transportador

,transportador,viagens,paletes,capacidade_total,taxa_media_viagem_pct,taxa_enchimento_pct
0,NO USAR,9,297.0,297.0,100.0,100.0
1,MOSCA PORTUGAL LDA,64,2112.0,2112.0,100.0,100.0
2,EL MOSCA S.A.,4,132.0,132.0,100.0,100.0
3,LOGISTA FREIGHT S.A.U.,41,1353.0,1353.0,100.0,100.0
4,SALVESEN,223,12411.25,12780.0,99.76,97.11
5,TORRESTIR - TRANSPORTES NACIONAIS E INTERNACIO...,180,8525.0,8987.0,99.51,94.86
6,TRANSPORTES CARGAKER LDA,27,1238.0,1408.0,85.04,87.93
7,JMR PRESTAÇAO DE SERVIÇOS PARA A DISTRIBUÇAO S.A,2071,69946.0,81493.0,85.8,85.83
8,TRANSPORTES PAULO COSTA & FERREIRA LDA,493,15876.0,18758.0,84.49,84.64
9,MODELO CONTINENTE HIPERMERCADOS S.A.,1396,42428.73,50420.0,83.82,84.15


In [22]:
# ==========================
# 7. Média global
# ==========================

taxa_enchimento_global_pct = (
    validacao_enchimento["paletes_ocupacao"].sum()
    / validacao_enchimento["capacidade"].sum()
    * 100
)

taxa_enchimento_global_pct

np.float64(76.12420085603156)

In [24]:

# ==========================
# 1. Consultar origem
# ==========================
import duckdb

with duckdb.connect(str(PATHS["db"]), read_only=True) as con:
    df_validacao_origem = con.execute("""
        WITH origem AS (
            SELECT
                SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6) AS MES,
                TRY_CAST(CODACT AS INTEGER) AS CODACT,
                NULLIF(TRIM(CAST(GESTION AS VARCHAR)), '') AS GESTION,
                TRIM(CAST(PROPIETARIO AS VARCHAR)) AS PROPIETARIO,
                NULLIF(TRIM(CAST(CODEUT AS VARCHAR)), '') AS CODEUT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(INGRESODT AS VARCHAR)), ',', '.')
                    AS DECIMAL(18, 2)
                ) AS INGRESODT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(COSTEDT AS VARCHAR)), ',', '.')
                    AS DECIMAL(18, 2)
                ) AS COSTEDT,

                TRY_CAST(
                    REPLACE(TRIM(CAST(PALETS AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PALETS,

                TRY_CAST(
                    REPLACE(TRIM(CAST(PESO_BRUTO AS VARCHAR)), ',', '.')
                    AS DOUBLE
                ) AS PESO_BRUTO

            FROM inform_27_2026
            WHERE SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
                  IN ('202607', '202608', '202609', '202610')
        ),

        classificada AS (
            SELECT *,
                CODACT IS NOT NULL
                AND CODACT <> 13
                AND (
                    GESTION = 'LIS'
                    OR (
                        GESTION IS NULL
                        AND PROPIETARIO IN ('CEP', 'PTG')
                    )
                ) AS FINANCEIRO
            FROM origem
        )

        SELECT
            MES,
            COUNT(*) AS LINHAS_RAW,
            COUNT(*) FILTER (WHERE FINANCEIRO) AS LINHAS_FINANCEIRO,
            COUNT(DISTINCT CODEUT) FILTER (WHERE FINANCEIRO) AS VIAGENS,

            ROUND(
                SUM(PALETS) FILTER (WHERE FINANCEIRO), 2
            ) AS PALETES,

            ROUND(
                SUM(PESO_BRUTO) FILTER (WHERE FINANCEIRO), 2
            ) AS PESO_KG,

            ROUND(SUM(INGRESODT), 2) AS INGRESSO_RAW,
            ROUND(SUM(COSTEDT), 2) AS CUSTO_RAW,

            ROUND(
                SUM(INGRESODT) FILTER (WHERE FINANCEIRO), 2
            ) AS INGRESSO_FINANCEIRO,

            ROUND(
                SUM(COSTEDT) FILTER (WHERE FINANCEIRO), 2
            ) AS CUSTO_FINANCEIRO

        FROM classificada
        GROUP BY MES
        ORDER BY MES
    """).df()

# ==========================
# 2. Resultado
# ==========================
df_validacao_origem


,MES,LINHAS_RAW,LINHAS_FINANCEIRO,VIAGENS,PALETES,PESO_KG,INGRESSO_RAW,CUSTO_RAW,INGRESSO_FINANCEIRO,CUSTO_FINANCEIRO
0,202607,100348,14127,2167,49017.50,13299174.52,3671504.30,8539053.70,406394.21,536690.18
1,202608,90440,13227,2020,49028.00,12434471.75,3173223.21,6567442.12,446345.62,499971.34
2,202609,15856,12713,1978,46723.00,12150463.39,435003.37,537188.04,435003.37,473748.50
3,202610,16618,13468,1989,46507.18,12285313.09,441068.86,558924.34,441068.86,497931.57
